In [2]:
import duckdb, numpy as np
from pathlib import Path
from transformers import AutoTokenizer

SEED = 2026
con = duckdb.connect()
PROCESSED = Path("..") / "data" / "processed"
path = (PROCESSED / "train_supervised.csv").as_posix()

sample = con.execute(f"""
    SELECT body_norm
    FROM read_csv('{path}', header=true, all_varchar=true)
    WHERE body_norm IS NOT NULL
    USING SAMPLE reservoir(20000 ROWS) REPEATABLE ({SEED})
""").df()
print(len(sample))

tok = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased")
enc = tok(sample["body_norm"].tolist(), add_special_tokens=True, truncation=False)
lengths = np.array([len(ids) for ids in enc["input_ids"]])

print(np.percentile(lengths, [50, 75, 90, 95, 99]))
for L in (32, 64, 128, 256, 512):
    print(L, round(100 * (lengths > L).mean(), 2), "% longer than", L)

20000
[ 9. 15. 27. 38. 71.]
32 7.0 % longer than 32
64 1.34 % longer than 64
128 0.16 % longer than 128
256 0.0 % longer than 256
512 0.0 % longer than 512
